In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
import tensorflow as tf

In [2]:
df = pd.read_csv("data/data_id_pintu.csv")
display(df)
# List semua ID pintu yang unik
unique_pintu = df['nilai'].unique()
unique_pintu

,nilai
0,107
1,114
2,116
3,118
4,126
5,140
6,150
7,155
8,158
9,160


array([107, 114, 116, 118, 126, 140, 150, 155, 158, 160, 164, 165, 166,
       167, 170, 172, 176, 178, 179, 181, 184, 187, 189, 191, 192, 193,
       194, 195, 196, 197, 198, 199, 200, 201])

In [3]:
df_list = []
for pintu_id in unique_pintu:
    print(f"Training model untuk pintu: {pintu_id}")
    df_data = pd.read_csv(f"data/data_{pintu_id}.csv")
    df_list.append(df_data)
    
# Menggabungkan semua DataFrame dalam daftar df_list secara vertikal
df_combined = pd.concat(df_list, ignore_index=True)
df_combined

Training model untuk pintu: 107
Training model untuk pintu: 114
Training model untuk pintu: 116
Training model untuk pintu: 118
Training model untuk pintu: 126
Training model untuk pintu: 140
Training model untuk pintu: 150
Training model untuk pintu: 155
Training model untuk pintu: 158
Training model untuk pintu: 160
Training model untuk pintu: 164
Training model untuk pintu: 165
Training model untuk pintu: 166
Training model untuk pintu: 167
Training model untuk pintu: 170
Training model untuk pintu: 172
Training model untuk pintu: 176
Training model untuk pintu: 178
Training model untuk pintu: 179
Training model untuk pintu: 181
Training model untuk pintu: 184
Training model untuk pintu: 187
Training model untuk pintu: 189
Training model untuk pintu: 191
Training model untuk pintu: 192
Training model untuk pintu: 193
Training model untuk pintu: 194
Training model untuk pintu: 195
Training model untuk pintu: 196
Training model untuk pintu: 197
Training model untuk pintu: 198
Training

,Unnamed: 0,timestamps,id_pintu_air,tinggi_air,status,temp,dwpt,rhum,prcp,snow,wdir,wspd,wpgt,pres,coco
0,0,2022-10-22 14:00:00,107.0,150.37,Normal,30.70,23.55,65.72,0.1,0.0,338.55,10.83,24.84,1007.85,90.0
1,1,2022-10-22 14:30:00,107.0,150.37,Normal,30.18,23.45,67.34,0.1,0.0,338.55,10.83,24.84,1007.85,90.0
2,2,2022-10-22 15:00:00,107.0,150.37,Normal,29.65,23.35,68.97,0.0,0.0,281.77,8.83,25.20,1007.85,99.0
3,3,2022-10-22 15:30:00,107.0,150.37,Normal,28.30,23.72,76.70,0.0,0.0,281.77,8.83,25.20,1007.80,99.0
4,4,2022-10-22 16:00:00,107.0,150.37,Normal,26.95,24.10,84.43,0.4,0.0,204.30,12.25,25.92,1007.74,100.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1218606,4773,2026-02-27 22:00:00,201.0,74.00,Normal,26.80,21.75,73.88,0.0,0.0,243.78,13.44,29.52,1007.28,100.0
1218607,4774,2026-02-27 22:30:00,201.0,74.00,Normal,26.60,21.85,75.22,0.0,0.0,243.78,13.44,29.52,1007.23,100.0
1218608,4775,2026-02-27 23:00:00,201.0,73.72,Normal,26.40,21.95,76.57,0.0,0.0,236.52,13.38,31.32,1007.18,100.0
1218609,4776,2026-02-27 23:30:00,201.0,73.00,Normal,26.32,21.95,76.91,0.0,0.0,236.52,13.38,31.32,1006.93,100.0


In [4]:
df_combined = df_combined.drop(columns=['Unnamed: 0'])
df_combined

,timestamps,id_pintu_air,tinggi_air,status,temp,dwpt,rhum,prcp,snow,wdir,wspd,wpgt,pres,coco
0,2022-10-22 14:00:00,107.0,150.37,Normal,30.70,23.55,65.72,0.1,0.0,338.55,10.83,24.84,1007.85,90.0
1,2022-10-22 14:30:00,107.0,150.37,Normal,30.18,23.45,67.34,0.1,0.0,338.55,10.83,24.84,1007.85,90.0
2,2022-10-22 15:00:00,107.0,150.37,Normal,29.65,23.35,68.97,0.0,0.0,281.77,8.83,25.20,1007.85,99.0
3,2022-10-22 15:30:00,107.0,150.37,Normal,28.30,23.72,76.70,0.0,0.0,281.77,8.83,25.20,1007.80,99.0
4,2022-10-22 16:00:00,107.0,150.37,Normal,26.95,24.10,84.43,0.4,0.0,204.30,12.25,25.92,1007.74,100.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1218606,2026-02-27 22:00:00,201.0,74.00,Normal,26.80,21.75,73.88,0.0,0.0,243.78,13.44,29.52,1007.28,100.0
1218607,2026-02-27 22:30:00,201.0,74.00,Normal,26.60,21.85,75.22,0.0,0.0,243.78,13.44,29.52,1007.23,100.0
1218608,2026-02-27 23:00:00,201.0,73.72,Normal,26.40,21.95,76.57,0.0,0.0,236.52,13.38,31.32,1007.18,100.0
1218609,2026-02-27 23:30:00,201.0,73.00,Normal,26.32,21.95,76.91,0.0,0.0,236.52,13.38,31.32,1006.93,100.0


In [5]:
from sklearn.preprocessing import LabelEncoder

# 1. Inisialisasi LabelEncoder
le_pintu = LabelEncoder()

# 2. Fit dan Transform pada kolom id_pintu_air di DataFrame utama (sebelum split)
# Misal nama dataframe utama Anda adalah all_data_df
df_combined['id_pintu_air_encoded'] = le_pintu.fit_transform(df_combined['id_pintu_air'])

In [6]:
df_combined = df_combined.sort_values(by=['id_pintu_air','timestamps'], ascending=[True,True])
df_combined.head()

,timestamps,id_pintu_air,tinggi_air,status,temp,dwpt,rhum,prcp,snow,wdir,wspd,wpgt,pres,coco,id_pintu_air_encoded
0,2022-10-22 14:00:00,107.0,150.37,Normal,30.70,23.55,65.72,0.1,0.0,338.55,10.83,24.84,1007.85,90.0,0
1,2022-10-22 14:30:00,107.0,150.37,Normal,30.18,23.45,67.34,0.1,0.0,338.55,10.83,24.84,1007.85,90.0,0
2,2022-10-22 15:00:00,107.0,150.37,Normal,29.65,23.35,68.97,0.0,0.0,281.77,8.83,25.20,1007.85,99.0,0
3,2022-10-22 15:30:00,107.0,150.37,Normal,28.30,23.72,76.70,0.0,0.0,281.77,8.83,25.20,1007.80,99.0,0
4,2022-10-22 16:00:00,107.0,150.37,Normal,26.95,24.10,84.43,0.4,0.0,204.30,12.25,25.92,1007.74,100.0,0


In [7]:
# Get the unique values from the 'kode_variasi_encoding' column
unique_variations = df_combined['id_pintu_air'].unique()

# Create an empty dictionary to store the dataframes
grouped_dataframes = {}

# Iterate through the unique values and create dataframes for each
for variation_code in unique_variations:
    grouped_dataframes[variation_code] = df_combined[df_combined['id_pintu_air'] == variation_code].copy()

# Display the keys of the dictionary to show the created dataframes
print("Created dataframes for the following variation codes:")
print(grouped_dataframes.keys())

Created dataframes for the following variation codes:
dict_keys([np.float64(107.0), np.float64(114.0), np.float64(116.0), np.float64(118.0), np.float64(126.0), np.float64(140.0), np.float64(150.0), np.float64(155.0), np.float64(158.0), np.float64(160.0), np.float64(164.0), np.float64(165.0), np.float64(166.0), np.float64(167.0), np.float64(170.0), np.float64(172.0), np.float64(176.0), np.float64(178.0), np.float64(179.0), np.float64(181.0), np.float64(184.0), np.float64(187.0), np.float64(189.0), np.float64(191.0), np.float64(192.0), np.float64(193.0), np.float64(194.0), np.float64(195.0), np.float64(196.0), np.float64(197.0), np.float64(198.0), np.float64(199.0), np.float64(200.0), np.float64(201.0)])


In [8]:
split_datasets = {}

for variation_code, df in grouped_dataframes.items():
    # Determine split points (e.g., 70% train, 15% validation, 15% test)
    total_rows = len(df)
    train_size = int(total_rows * 0.7)
    validation_size = int(total_rows * 0.15)
    # The remaining rows will be for testing

    # Split the dataframe chronologically
    train_df = df.iloc[:train_size]
    validation_df = df.iloc[train_size : train_size + validation_size]
    test_df = df.iloc[train_size + validation_size :]

    # Store the split datasets
    split_datasets[variation_code] = {
        'train': train_df,
        'validation': validation_df,
        'test': test_df
    }

# Display the keys of the split_datasets to show the created splits per variation
print("Split datasets created for the following variation codes:")
print(split_datasets.keys())

Split datasets created for the following variation codes:
dict_keys([np.float64(107.0), np.float64(114.0), np.float64(116.0), np.float64(118.0), np.float64(126.0), np.float64(140.0), np.float64(150.0), np.float64(155.0), np.float64(158.0), np.float64(160.0), np.float64(164.0), np.float64(165.0), np.float64(166.0), np.float64(167.0), np.float64(170.0), np.float64(172.0), np.float64(176.0), np.float64(178.0), np.float64(179.0), np.float64(181.0), np.float64(184.0), np.float64(187.0), np.float64(189.0), np.float64(191.0), np.float64(192.0), np.float64(193.0), np.float64(194.0), np.float64(195.0), np.float64(196.0), np.float64(197.0), np.float64(198.0), np.float64(199.0), np.float64(200.0), np.float64(201.0)])


In [9]:
# Initialize empty dataframes for the combined splits
all_train_df = pd.DataFrame()
all_validation_df = pd.DataFrame()
all_test_df = pd.DataFrame()

# Iterate through the split_datasets and concatenate the dataframes
for variation_code, splits in split_datasets.items():
    all_train_df = pd.concat([all_train_df, splits['train']], ignore_index=True)
    all_validation_df = pd.concat([all_validation_df, splits['validation']], ignore_index=True)
    all_test_df = pd.concat([all_test_df, splits['test']], ignore_index=True)

# Display the head of the combined dataframes to verify
print("Combined Training Data:")
display(all_train_df.head())
print("\nCombined Validation Data:")
display(all_validation_df.head())
print("\nCombined Test Data:")
display(all_test_df.head())

Combined Training Data:


,timestamps,id_pintu_air,tinggi_air,status,temp,dwpt,rhum,prcp,snow,wdir,wspd,wpgt,pres,coco,id_pintu_air_encoded
0,2022-10-22 14:00:00,107.0,150.37,Normal,30.70,23.55,65.72,0.1,0.0,338.55,10.83,24.84,1007.85,90.0,0
1,2022-10-22 14:30:00,107.0,150.37,Normal,30.18,23.45,67.34,0.1,0.0,338.55,10.83,24.84,1007.85,90.0,0
2,2022-10-22 15:00:00,107.0,150.37,Normal,29.65,23.35,68.97,0.0,0.0,281.77,8.83,25.20,1007.85,99.0,0
3,2022-10-22 15:30:00,107.0,150.37,Normal,28.30,23.72,76.70,0.0,0.0,281.77,8.83,25.20,1007.80,99.0,0
4,2022-10-22 16:00:00,107.0,150.37,Normal,26.95,24.10,84.43,0.4,0.0,204.30,12.25,25.92,1007.74,100.0,0



Combined Validation Data:


,timestamps,id_pintu_air,tinggi_air,status,temp,dwpt,rhum,prcp,snow,wdir,wspd,wpgt,pres,coco,id_pintu_air_encoded
0,2025-02-25 03:30:00,107.0,223.66,Waspada,23.92,22.90,94.00,0.2,0.0,239.04,3.15,12.96,1011.58,100.0,0
1,2025-02-25 04:00:00,107.0,228.66,Waspada,23.95,22.95,94.15,0.1,0.0,250.35,2.68,8.28,1011.63,100.0,0
2,2025-02-25 04:30:00,107.0,235.15,Waspada,23.98,22.98,94.15,0.1,0.0,250.35,2.68,8.28,1011.68,100.0,0
3,2025-02-25 05:00:00,107.0,241.49,Waspada,24.00,23.00,94.15,0.2,0.0,255.96,3.71,10.44,1011.73,100.0,0
4,2025-02-25 05:30:00,107.0,247.32,Waspada,23.90,23.02,94.87,0.2,0.0,255.96,3.71,10.44,1011.93,100.0,0



Combined Test Data:


,timestamps,id_pintu_air,tinggi_air,status,temp,dwpt,rhum,prcp,snow,wdir,wspd,wpgt,pres,coco,id_pintu_air_encoded
0,2025-08-27 16:30:00,107.0,240.84,Waspada,31.72,19.15,47.49,0.0,0.0,354.81,13.92,39.60,1007.20,100.0,0
1,2025-08-27 17:00:00,107.0,240.00,Waspada,31.00,19.75,51.21,0.0,0.0,339.62,13.44,35.64,1007.45,100.0,0
2,2025-08-27 17:30:00,107.0,239.17,Waspada,30.42,20.22,54.61,0.0,0.0,339.62,13.44,35.64,1007.80,100.0,0
3,2025-08-27 18:00:00,107.0,240.32,Waspada,29.85,20.70,58.01,0.0,0.0,324.93,10.34,32.76,1008.15,100.0,0
4,2025-08-27 18:30:00,107.0,241.16,Waspada,29.70,21.10,60.00,0.0,0.0,324.93,10.34,32.76,1008.60,100.0,0


In [10]:
from sklearn.preprocessing import MinMaxScaler
import joblib

fitur = [
    'temp',
    'pres',
    'rhum',
    'wdir',
    'wpgt'
]
scaler_features = MinMaxScaler()

all_train_df[fitur] = scaler_features.fit_transform(all_train_df[fitur])
all_train_df

,timestamps,id_pintu_air,tinggi_air,status,temp,dwpt,rhum,prcp,snow,wdir,wspd,wpgt,pres,coco,id_pintu_air_encoded
0,2022-10-22 14:00:00,107.0,150.37,Normal,0.605327,23.55,0.573048,0.1,0.0,0.940317,10.83,0.374302,0.873463,90.0,0
1,2022-10-22 14:30:00,107.0,150.37,Normal,0.580145,23.45,0.593225,0.1,0.0,0.940317,10.83,0.374302,0.873463,90.0,0
2,2022-10-22 15:00:00,107.0,150.37,Normal,0.554479,23.35,0.613526,0.0,0.0,0.782332,8.83,0.379888,0.873463,99.0,0
3,2022-10-22 15:30:00,107.0,150.37,Normal,0.489104,23.72,0.709802,0.0,0.0,0.782332,8.83,0.379888,0.872644,99.0,0
4,2022-10-22 16:00:00,107.0,150.37,Normal,0.423729,24.10,0.806078,0.4,0.0,0.566778,12.25,0.391061,0.871660,100.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
853005,2026-01-29 01:00:00,201.0,98.67,Normal,0.273608,23.40,0.966746,1.6,0.0,0.416973,6.21,0.240223,0.924603,100.0,33
853006,2026-01-29 01:30:00,201.0,99.33,Normal,0.266344,23.12,0.957529,1.6,0.0,0.416973,6.21,0.240223,0.919685,100.0,33
853007,2026-01-29 02:00:00,201.0,99.67,Normal,0.259080,22.85,0.948437,0.5,0.0,0.568114,5.15,0.301676,0.914768,100.0,33
853008,2026-01-29 02:30:00,201.0,97.67,Normal,0.241162,22.65,0.961141,0.5,0.0,0.568114,5.15,0.301676,0.913129,100.0,33


In [11]:
all_validation_df[fitur] = scaler_features.fit_transform(all_validation_df[fitur])

In [12]:
all_test_df[fitur] = scaler_features.fit_transform(all_test_df[fitur])

In [13]:
scalers = {}
list_scaled_train_dfs = []

# 1. Pastikan kolom target dalam bentuk list jika belum
target_col = ['tinggi_air'] # Sesuaikan dengan nama kolom target Anda

# 2. Iterasi melalui hasil groupby
for pintu_id, df_group in all_train_df.groupby('id_pintu_air'):
    # Gunakan .copy() agar tidak terkena SettingWithCopyWarning
    df_temp = df_group.copy()
    
    # Inisialisasi scaler untuk ID pintu air ini
    scaler_target = MinMaxScaler()
    
    # Fit & Transform hanya pada kolom target
    # Kita gunakan df_temp[target_col] agar tetap dalam format 2D (DataFrame/Numpy)
    df_temp[target_col] = scaler_target.fit_transform(df_temp[target_col])
    
    # Simpan scaler ke dictionary menggunakan ID sebagai key
    scalers[pintu_id] = scaler_target
    
    # Masukkan DataFrame yang sudah di-scale ke list baru
    list_scaled_train_dfs.append(df_temp)

# Hasil akhirnya adalah list berisi DataFrame yang sudah ter-scale per pintu air
list_of_train_dfs = list_scaled_train_dfs

In [14]:
list_scaled_val_dfs = []
# 2. Iterasi melalui hasil groupby
for pintu_id, df_group in all_validation_df.groupby('id_pintu_air'):
    # Gunakan .copy() agar tidak terkena SettingWithCopyWarning
    df_temp = df_group.copy()
    
    # Inisialisasi scaler untuk ID pintu air ini
    
    # Fit & Transform hanya pada kolom target
    # Kita gunakan df_temp[target_col] agar tetap dalam format 2D (DataFrame/Numpy)
    df_temp[target_col] = scalers[pintu_id].fit_transform(df_temp[target_col])
    
    
    # Masukkan DataFrame yang sudah di-scale ke list baru
    list_scaled_val_dfs.append(df_temp)

# Hasil akhirnya adalah list berisi DataFrame yang sudah ter-scale per pintu air
list_of_validation_dfs = list_scaled_val_dfs

In [15]:
list_scaled_test_dfs = []
# 2. Iterasi melalui hasil groupby
for pintu_id, df_group in all_test_df.groupby('id_pintu_air'):
    # Gunakan .copy() agar tidak terkena SettingWithCopyWarning
    df_temp = df_group.copy()
    
    # Inisialisasi scaler untuk ID pintu air ini
    
    # Fit & Transform hanya pada kolom target
    # Kita gunakan df_temp[target_col] agar tetap dalam format 2D (DataFrame/Numpy)
    df_temp[target_col] = scalers[pintu_id].fit_transform(df_temp[target_col])
    
    
    # Masukkan DataFrame yang sudah di-scale ke list baru
    list_scaled_test_dfs.append(df_temp)

# Hasil akhirnya adalah list berisi DataFrame yang sudah ter-scale per pintu air
list_of_test_dfs = list_scaled_test_dfs

In [16]:
import numpy as np

def create_sequences(data_variasi, data_exogen, data_target, time_step):
    X_var, X_exo, Y = [], [], []

    # Pastikan semua data memiliki panjang yang sama
    assert len(data_variasi) == len(data_exogen) == len(data_target)

    for i in range(len(data_variasi) - time_step):
        # 1. Input Variasi (Array 2D)
        # Ambil sekuens integer ID variasi dari i hingga i + time_step
        seq_var = data_variasi[i : i + time_step]
        X_var.append(seq_var)

        # 2. Input Eksogen (Array 3D)
        # Ambil sekuens fitur eksogen dari i hingga i + time_step
        seq_exo = data_exogen[i : i + time_step, :]
        X_exo.append(seq_exo)

        # 3. Output Target (Nilai pada waktu t)
        # Ambil nilai target berikutnya
        target_val = data_target[i + time_step]
        Y.append(target_val)

    return np.array(X_var), np.array(X_exo), np.array(Y)

In [17]:
import numpy as np

fitur_eksogen_scaled = [
    'temp',
    'pres',
    'rhum',
    'wdir',
    'wpgt',
    'tinggi_air'
]
# Daftar untuk menampung hasil sekuens dari setiap variasi
X_var_list, X_exo_list, Y_list = [], [], []
TIME_STEP = 48

for df_variasi in list_of_train_dfs:
    # Pisahkan kolom variasi ID, eksogen, dan target dari df_variasi
    data_var = df_variasi['id_pintu_air_encoded'].values
    data_exo = df_variasi[fitur_eksogen_scaled].values
    data_target = df_variasi['tinggi_air'].values

    # Buat sekuens
    X_var_i, X_exo_i, Y_i = create_sequences(data_var, data_exo, data_target, TIME_STEP)

    # Tambahkan ke list
    X_var_list.append(X_var_i)
    X_exo_list.append(X_exo_i)
    Y_list.append(Y_i)

# Gabungkan semua hasil sekuens menjadi array NumPy tunggal
X_var_train = np.concatenate(X_var_list)
X_exo_train = np.concatenate(X_exo_list)
y_train = np.concatenate(Y_list)

In [18]:
print(X_exo_train.shape)
print(y_train.shape)

(851378, 48, 6)
(851378,)


In [19]:
# Daftar untuk menampung hasil sekuens dari setiap variasi
X_var_list, X_exo_list, Y_list = [], [], []

# Lakukan loop pada setiap DataFrame variasi
for df_variasi in list_of_validation_dfs:
    # Pisahkan kolom variasi ID, eksogen, dan target dari df_variasi
    data_var = df_variasi['id_pintu_air_encoded'].values
    data_exo = df_variasi[fitur_eksogen_scaled].values
    data_target = df_variasi['tinggi_air'].values

    # Buat sekuens
    X_var_i, X_exo_i, Y_i = create_sequences(data_var, data_exo, data_target, TIME_STEP)

    # Tambahkan ke list
    X_var_list.append(X_var_i)
    X_exo_list.append(X_exo_i)
    Y_list.append(Y_i)

# Gabungkan semua hasil sekuens menjadi array NumPy tunggal
X_var_val = np.concatenate(X_var_list)
X_exo_val= np.concatenate(X_exo_list)
y_val = np.concatenate(Y_list)

In [20]:
# Daftar untuk menampung hasil sekuens dari setiap variasi
X_var_list, X_exo_list, Y_list = [], [], []

# Lakukan loop pada setiap DataFrame variasi
for df_variasi in list_of_test_dfs:
    # Pisahkan kolom variasi ID, eksogen, dan target dari df_variasi
    data_var = df_variasi['id_pintu_air_encoded'].values
    data_exo = df_variasi[fitur_eksogen_scaled].values
    data_target = df_variasi['tinggi_air'].values

    # Buat sekuens
    X_var_i, X_exo_i, Y_i = create_sequences(data_var, data_exo, data_target, TIME_STEP)

    # Tambahkan ke list
    X_var_list.append(X_var_i)
    X_exo_list.append(X_exo_i)
    Y_list.append(Y_i)

# Gabungkan semua hasil sekuens menjadi array NumPy tunggal
X_var_test = np.concatenate(X_var_list)
X_exo_test= np.concatenate(X_exo_list)
y_test = np.concatenate(Y_list)

In [25]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Embedding, LSTM, Dense, 
                                     concatenate, Dropout, AdditiveAttention, 
                                     Permute, Reshape, Multiply, Flatten)

# --- 1. Definisi Hyperparameter ---
TIME_STEP = 48 
MAX_VARIATIONS = int(all_train_df['id_pintu_air_encoded'].nunique()) + 1 
NUM_EXOGENOUS_FEATURES = len(fitur_eksogen_scaled) 
EMBEDDING_DIM = 8 

# --- 2. Jalur 1: Embedding untuk Kode Variasi ---
input_variasi = Input(shape=(TIME_STEP,), dtype='int32', name='variasi_sequence_input')
embedding = Embedding(
    input_dim=MAX_VARIATIONS,
    output_dim=EMBEDDING_DIM,
    name='variasi_embedding'
)(input_variasi) 

lstm_variasi = LSTM(64, activation='relu', name='lstm_variasi')(embedding) 
output_jalur_variasi = Dropout(0.2)(lstm_variasi)

# --- 3. Jalur 2: LSTM + ATTENTION untuk Fitur Eksogen ---
input_exogen = Input(shape=(TIME_STEP, NUM_EXOGENOUS_FEATURES), name='exogen_features_input')

# LSTM pertama harus return_sequences=True agar bisa diproses Attention
lstm_exogen_1 = LSTM(
    units=64,
    activation='relu',
    return_sequences=True, 
    name='lstm_exogen_1'
)(input_exogen)
lstm_exogen_1 = Dropout(0.2, name='dropout_exogen_1')(lstm_exogen_1)

lstm_exogen_2 = LSTM(
    units=64,
    activation='relu',
    return_sequences=True, 
    name='lstm_exogen_2'
)(lstm_exogen_1) 

lstm_exogen_2 = Dropout(0.2, name='dropout_exogen_2')(lstm_exogen_2)

# --- MEKANISME ATTENTION ---
# 1. Kita buat 'query' dan 'value' untuk Attention. 
# Di sini kita menggunakan output LSTM itu sendiri sebagai query dan value.
query = Dense(64)(lstm_exogen_2) 
value = Dense(64)(lstm_exogen_2)

# 2. Lapisan Attention
attention_vct = AdditiveAttention(name='attention_layer')([query, value])

# 3. Gabungkan (Reduce) informasi Attention
# Karena kita ingin satu vektor ringkasan, kita bisa menggunakan GlobalAveragePooling atau Flatten
attention_output = tf.keras.layers.GlobalAveragePooling1D()(attention_vct)

# Jalur LSTM kedua (Opsional, atau langsung gunakan attention_output)
output_jalur_exogen = Dropout(0.2)(attention_output)

# --- 4. Penggabungan dan Output ---
gabungan = concatenate([output_jalur_variasi, output_jalur_exogen], name='gabungan_fitur')

dense = Dense(64, activation='relu', name='dense_pemrosesan_akhir')(gabungan)
dense = Dropout(0.2)(dense)
output = Dense(1, activation='linear', name='output_tinggi_air_scaled')(dense)

# --- 5. Definisi Model ---
model = Model(inputs=[input_variasi, input_exogen], outputs=output)

model.compile(
    optimizer='adam',
    loss='mean_squared_error',
    metrics=[tf.keras.metrics.MeanSquaredError(name='mse'),
        tf.keras.metrics.RootMeanSquaredError(name='rmse'),
        tf.keras.metrics.MeanAbsoluteError(name='mae')]
)

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ exogen_features_input         │ (None, 48, 6)             │               0 │ -                          │
│ (InputLayer)                  │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ lstm_exogen_1 (LSTM)          │ (None, 48, 64)            │          18,176 │ exogen_features_input[0][… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout_exogen_1 (Dropout)    │ (None, 48, 64)            │               0 │ lstm_exogen_1[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ lstm_exogen_2 (LSTM)          │ (None, 48, 64)            │          33,024 │ dropout_exogen_1[0][0]     │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout_exogen_2 (Dropout)    │ (None, 48, 64)            │               0 │ lstm_exogen_2[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ variasi_sequence_input        │ (None, 48)                │               0 │ -                          │
│ (InputLayer)                  │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_4 (Dense)               │ (None, 48, 64)            │           4,160 │ dropout_exogen_2[0][0]     │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_5 (Dense)               │ (None, 48, 64)            │           4,160 │ dropout_exogen_2[0][0]     │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ variasi_embedding (Embedding) │ (None, 48, 8)             │             280 │ variasi_sequence_input[0]… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ attention_layer               │ (None, 48, 64)            │              64 │ dense_4[0][0],             │
│ (AdditiveAttention)           │                           │                 │ dense_5[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ lstm_variasi (LSTM)           │ (None, 64)                │          18,688 │ variasi_embedding[0][0]    │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ global_average_pooling1d_2    │ (None, 64)                │               0 │ attention_layer[0][0]      │
│ (GlobalAveragePooling1D)      │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout_4 (Dropout)           │ (None, 64)                │               0 │ lstm_variasi[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout_5 (Dropout)           │ (None, 64)                │               0 │ global_average_pooling1d_… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gabungan_fitur (Concatenate)  │ (None, 128)               │               0 │ dropout_4[0][0],           │
│                               │                           │               

 Total params: 86,873 (339.35 KB)

 Trainable params: 86,873 (339.35 KB)

 Non-trainable params: 0 (0.00 B)

In [26]:
from tensorflow.keras.callbacks import EarlyStopping

# Tambahkan EarlyStopping
early_stop = EarlyStopping(
    monitor='val_loss',     # pantau validation loss
    patience=5,             # hentikan training jika val_loss tidak membaik selama 5 epoch
    restore_best_weights=True,  # kembalikan bobot terbaik
    verbose=1
)

In [ ]:
history = model.fit(
    [X_var_train, X_exo_train], # Input Ganda
    y_train,
    epochs=50,
    batch_size=128,
    validation_data=([X_var_val, X_exo_val], y_val),
    verbose=1
)

Epoch 1/50
6652/6652 ━━━━━━━━━━━━━━━━━━━━ 4045s 605ms/step - loss: 0.0051 - mae: 0.0425 - mse: 0.0051 - rmse: 0.0691 - val_loss: 0.0024 - val_mae: 0.0259 - val_mse: 0.0024 - val_rmse: 0.0488
Epoch 2/50
6652/6652 ━━━━━━━━━━━━━━━━━━━━ 5896s 886ms/step - loss: 0.0014 - mae: 0.0225 - mse: 0.0014 - rmse: 0.0378 - val_loss: 0.0027 - val_mae: 0.0294 - val_mse: 0.0027 - val_rmse: 0.0517
Epoch 3/50
6652/6652 ━━━━━━━━━━━━━━━━━━━━ 5794s 869ms/step - loss: 0.0013 - mae: 0.0207 - mse: 0.0013 - rmse: 0.0356 - val_loss: 0.0027 - val_mae: 0.0298 - val_mse: 0.0027 - val_rmse: 0.0520
Epoch 4/50
6652/6652 ━━━━━━━━━━━━━━━━━━━━ 5201s 782ms/step - loss: 0.0012 - mae: 0.0198 - mse: 0.0012 - rmse: 0.0343 - val_loss: 0.0039 - val_mae: 0.0368 - val_mse: 0.0039 - val_rmse: 0.0627
Epoch 5/50
6652/6652 ━━━━━━━━━━━━━━━━━━━━ 3596s 540ms/step - loss: 0.0011 - mae: 0.0194 - mse: 0.0011 - rmse: 0.0336 - val_loss: 0.0039 - val_mae: 0.0400 - val_mse: 0.0039 - val_rmse: 0.0627
Epoch 6/50
6652/6652 ━━━━━━━━━━━━━━━━━━━━ 329

In [ ]:
test_results = model.evaluate(
    [X_var_test, X_exo_test],  # Input ganda
    y_test,
    verbose=1
)
print(test_results)

In [ ]:
# Lakukan prediksi ulang untuk memastikan prediksi_scaled tersedia
prediksi_scaled = model.predict([X_var_test, X_exo_test])

In [ ]:
first_elements = X_var_test[:, 0]
first_elements

In [ ]:
prediksi_scaled